# Pipeline

A pipeline chains steps together so they run in order, automatically.

Instead of running each step separately, you package them into one object and call `.fit()` once.


#### Example Data

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

data = {
    'hours_studied': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'exam_score':    [40, 45, 55, 58, 62, 70, 75, 82, 88, 95],
    'passed':        [0,  0,  0,  0,  1,  1,  1,  1,  1,  1],
}
df = pd.DataFrame(data)

X = df[['hours_studied', 'exam_score']]
y = df['passed']

df

,hours_studied,exam_score,passed
0,1,40,0
1,2,45,0
2,3,55,0
3,4,58,0
4,5,62,1
5,6,70,1
6,7,75,1
7,8,82,1
8,9,88,1
9,10,95,1



## Without a Pipeline

You have to run each step manually — and you must remember the right order every time.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 1 — scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)    # transform only — not fit!

# Step 2 — train
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Step 3 — evaluate
print('Score:', model.score(X_test_scaled, y_test))

Score: 1.0


Every time you get new data, you have to remember to scale it first.


## With a Pipeline

Package all the steps into one object. Same result but cleaner and safer.

In [3]:
pipe = Pipeline([
    ('scale', StandardScaler()),      # step 1 — standardise
    ('model', LogisticRegression()),  # step 2 — train
])

pipe.fit(X_train, y_train)

print('Score:', pipe.score(X_test, y_test))

Score: 1.0


The pipeline handles the scaling automatically — including on new data.


## Predicting a New Student

In [ ]:
new_student = pd.DataFrame({
    'hours_studied': [6],
    'exam_score':    [72],
})

result = pipe.predict(new_student)
print('Predicted:', 'Pass' if result[0] == 1 else 'Fail')
# Pipeline scales the new data first, then predicts — automatically

Predicted: Pass



## Cross-Validation

Pass the whole pipeline into `cross_val_score` — each fold is scaled and trained correctly with no leakage.

- Cross-validation splits the data into 3 parts — each part gets a turn to be tested while the other 2 are used for training.

- Pass the whole pipeline so every test run includes all the steps from scratch — no leakage.

- cv=3 runs 3 tests. Scores: [0.75, 1.0, 0.67] — mean: 0.81

In [ ]:
scores = cross_val_score(pipe, X, y, cv=3)

print('CV scores:', scores.round(2))
print('Mean:     ', scores.mean().round(2))

# The mean 0.81 is the average of all 3 scores.

CV scores: [0.75 1.   0.67]
Mean:      0.81



## Adding More Steps

You can add any step from the series into the pipeline — just add another line.

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

pipe_full = Pipeline([
    ('impute',  SimpleImputer(strategy='mean')),
    ('scale',   StandardScaler()),
    ('extract', PCA(n_components=2)),
    ('model',   LogisticRegression()),
])

pipe_full.fit(X_train, y_train)
print('Score:', pipe_full.score(X_test, y_test))

Score: 1.0


---
## Summary

| | Without Pipeline | With Pipeline |
|---|---|---|
| Steps | Run manually, one by one | Run automatically, in order |
| New data | Must remember to transform | Handled automatically |
| Cross-validation | Leakage risk | Leak-free |
| Code | Long | Short |

# The End